# Hadith thesis: transformer runs (post-freeze supplement)

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. In Google Drive, create the folder `hadith_v2` in My Drive and put `hadith_v2_colab_pack.zip` in it.
3. Runtime > **Run all**, and approve the Drive permission prompt. Keep this tab open.

**If the session drops:** reconnect and use Run all again. Finished seeds are skipped and nothing is lost.

**Do not change any setting in these cells.** Hyperparameters are frozen (thesis/FREEZE_PROTOCOL.md).

## 1. Check the GPU, mount Drive, unpack and verify the data

In [ ]:
import os, sys, glob, json, hashlib, subprocess, zipfile
import pandas as pd

gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True)
assert gpu.returncode == 0, "NO GPU. Runtime > Change runtime type > T4 GPU, then Run all."
print(gpu.stdout)

from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/hadith_v2"
OUT = f"{DRIVE}/outputs"
WORK = "/content/hadith_v2"
PACK = f"{DRIVE}/hadith_v2_colab_pack.zip"
assert os.path.exists(PACK), f"Missing {PACK}: upload the zip to My Drive/hadith_v2/ first."
os.makedirs(OUT, exist_ok=True)
zipfile.ZipFile(PACK).extractall(WORK)

EXPECTED = {
    "binary/processed_train.csv": "17f8ad93086fbe50054ab026f25781eaac4ed10068066180652c5a468ed64fe1",
    "binary/processed_val.csv": "5809c659f66ca675f71c4cb2e0a6039e8861c20eccd55c7c3fce529af29a2ca5",
    "binary/processed_test.csv": "5632272231581c0a2bcfc9777756602f93a4d13a608cb177b810527f9a9e1bfe",
    "ternary/processed_train.csv": "a0e00d5b735935ec06ca38a2cb963071bb36f618592392956dfbf3a1b6f89750",
    "ternary/processed_val.csv": "818a73548ad9c32a761d5ceb5a352fadb3697c79f58460a46e1bbc1998f7cfc0",
    "ternary/processed_test.csv": "2292c7a63f8c450fa9050bba306c55adff1b07aa92419c8b35c087c5ebb121e4"
}
def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
for name, digest in EXPECTED.items():
    assert sha(f"{WORK}/{name}") == digest, f"{name} differs from the frozen data. Stop and tell Claude."
print("Data verified: all 6 files match the freeze manifest.")

## 2. Install the pinned libraries

In [ ]:
# Versions tested end to end with this script. A preinstalled peft can break the Trainer import and is not needed.
!pip -q uninstall -y peft
!pip -q install "transformers==4.51.3" "accelerate==1.15.0" "huggingface_hub==0.36.2" safetensors

ENV = dict(os.environ, USE_TF="0", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", HF_DATASETS_DISABLE_PROGRESS_BARS="1")

def run(task, model, log, *extra):
    cmd = [sys.executable, "train_transformer_v2.py", "--task", task, "--model", model, "--data_dir", task, *extra]
    p = subprocess.Popen(cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(log, "a") as f:
        for line in p.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
    assert p.wait() == 0, f"Run failed: {task} {model}. Send Claude the last 30 lines above."

MODELS = ["aubmindlab/bert-base-arabertv02", "CAMeL-Lab/bert-base-arabic-camelbert-ca"]
print("Libraries installed.")

## 3. Smoke test (about 3 minutes): proves both models run before the long job starts

In [ ]:
for model in MODELS:
    run("ternary", model, "/content/smoke.log", "--seeds", "0", "--limit", "200", "--epochs", "1",
        "--out_dir", "/content/smoke", "--work_dir", "/content/smoke_work")
print("SMOKE TEST PASSED. These numbers are meaningless and are not used.")

## 4. Full run: 2 models x 2 tasks x 5 seeds (20 trainings)

In [ ]:
for task in ["binary", "ternary"]:
    for model in MODELS:
        print(f"===== {task} | {model} =====", flush=True)
        run(task, model, f"{OUT}/log_{task}_{model.split('/')[-1]}.txt",
            "--seeds", "0", "1", "2", "3", "4", "--out_dir", OUT, "--work_dir", "/content/work")
print("ALL RUNS FINISHED.")

## 5. Check completeness and package the outputs

In [ ]:
missing = []
for task in ["binary", "ternary"]:
    for model in MODELS:
        short = f"{task}_{model.split('/')[-1]}"
        for s in range(5):
            for f in [f"{OUT}/val_probs_{short}_s{s}.npy", f"{OUT}/sealed_test_preds/test_probs_{short}_s{s}.npy"]:
                if not os.path.exists(f):
                    missing.append(os.path.basename(f))
print("MISSING:", missing if missing else "none, 40 probability files present")

!pip freeze > "{OUT}/pip_freeze.txt"
!nvidia-smi > "{OUT}/nvidia_smi.txt"

zip_out = f"{DRIVE}/hadith_v2_outputs.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/**", recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUT))
print("Saved:", zip_out, round(os.path.getsize(zip_out) / 1e6, 1), "MB")

# Validation only. Test probabilities stay sealed.
cols = ["task", "model", "macro_f1", "mawdu_f1", "accuracy"]
df = pd.concat(pd.read_csv(f) for f in glob.glob(f"{OUT}/results_transformer_v2_*.csv"))
print(df[cols + ["seed"]].groupby(["task", "model"]).agg(seeds=("seed", "nunique"), macro_f1=("macro_f1", "mean"), macro_sd=("macro_f1", "std"),
      mawdu_f1=("mawdu_f1", "mean"), mawdu_sd=("mawdu_f1", "std")).round(4).to_string())
drive.flush_and_unmount()